# Demo C — A text compressor built from a temporal autoencoder

**Class 11 · C1–C2** (slides: `slides/11_autoencoders.md`)

* **C1:** a 1-D convolutional autoencoder (`faa.textae`) reads windows of 32 characters and stores each window as 32 numbers of 4 bits (16 bytes). It is trained on four text files (BBC news: business, tech, politics, sport) and compresses a file it has never seen (entertainment). The same model runs from the command line: `python -m faa.textae train | compress | decompress`.
* **C2:** the lossy code against the lossless tools (gzip, bzip2, lzma), the code size against the accuracy, and failure cases (a different kind of text).

In [ ]:
import bz2
import gzip
import lzma
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import polars as pl

from faa import data, demo, plotlib, textae
from faa.demo import show, slide

demo.setup("11")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)

## Several text files

The BBC news set (1000 articles, five topics) is written as one `.txt` file per topic in a temporary folder. Four files train the model; the fifth (entertainment) is the unseen file to compress.

In [ ]:
bbc = pl.read_csv(data.data_dir() / "bbc_news.csv.zst")
work = Path(tempfile.mkdtemp(prefix="textae_"))
for topic in bbc["category"].unique().to_list():
    (work / f"bbc_{topic}.txt").write_text(
        "\n".join(bbc.filter(pl.col("category") == topic)["text"].to_list()), encoding="utf-8"
    )
train_files = sorted(work.glob("bbc_*.txt"))
test_file = work / "bbc_entertainment.txt"
train_files = [f for f in train_files if f != test_file]
train_texts = [f.read_text(encoding="utf-8") for f in train_files]
test_text = test_file.read_text(encoding="utf-8")[:20000]
(work / "test.txt").write_text(test_text, encoding="utf-8")
slide("The Text Compressor: Data")
for f in train_files:
    print(f"{f.name}: {f.stat().st_size} bytes")
print(
    f"unseen file (first 20000 characters of {test_file.name}): {len(test_text)} characters; the alphabet has {textae.V} symbols"
)

## Slide: Windows and Bytes (C1)

A window of $L = 32$ characters is $32 \times 8 = 256$ bits. The code has 32 units of 4 bits = 128 bits = 16 bytes: a ratio of 2 (before the 6-byte header of the file).

In [ ]:
w = textae.encode_text(test_text)
slide("Windows and Bytes")
print(f"{len(test_text)} characters -> {w.shape[0]} windows of {textae.L} characters")
print(
    f"code: 32 units x 4 bits = {32 * 4 // 8} bytes per window; file: {w.shape[0] * 16} bytes + {textae.HEADER.size} header = {w.shape[0] * 16 + textae.HEADER.size}"
)
print(f"ratio of the sizes: {len(test_text.encode()) / (w.shape[0] * 16 + textae.HEADER.size):.2f}")

## Slide: Training the Temporal Autoencoder (C1)

Convolutional encoder (two layers with stride 2: 32 characters to 8 steps), 4-bit code of 32 units, transposed-convolution decoder, softmax over the 96 symbols. 60 000 random windows of the four training files, 15 epochs in three cells (so that each cell stays short). Noise of half a quantisation step is added to the code while training.

In [ ]:
X = textae.windows(train_texts, 60000, 0)
ae, enc, dec = textae.build(32, 64, 0, "conv", 4)
hist = []
h = ae.fit(X, X, epochs=5, batch_size=128, shuffle=True, verbose=0)
hist += h.history["accuracy"]
print("character accuracy per epoch (train):", [round(a, 3) for a in hist])

In [ ]:
h = ae.fit(X, X, epochs=5, batch_size=128, shuffle=True, verbose=0)
hist += h.history["accuracy"]
print("epochs 6-10:", [round(a, 3) for a in hist[5:]])

In [ ]:
h = ae.fit(X, X, epochs=5, batch_size=128, shuffle=True, verbose=0)
hist += h.history["accuracy"]
slide("Training the Temporal Autoencoder")
print("epochs 11-15:", [round(a, 3) for a in hist[10:]])
fig, ax = plotlib.fig("col", 1.5)
ax.plot(range(1, len(hist) + 1), hist, color=BLUE, marker="o", ms=3)
ax.set_xlabel("epoch")
ax.set_ylabel("characters rebuilt (train)")
plotlib.save(fig, "c1_training")

## Slide: Compress and Decompress an Unseen File (C1)

The unseen text goes through the encoder (code rounded to 4 bits), the bytes are the compressed file, the decoder reads them back. The measure is the share of characters that are identical.

In [ ]:
blob = textae.compress(test_text, enc, 4)
back = textae.decompress(blob, dec)
slide("Compress and Decompress an Unseen File")
print(f"{len(test_text.encode())} bytes -> {len(blob)} bytes: ratio {len(test_text.encode()) / len(blob):.2f}")
print(f"characters identical after the round trip: {textae.char_accuracy(test_text, back):.3f}")
words_t, words_b = test_text.split(), back.split()
print(
    f"whole words rebuilt exactly: {np.mean([a == b for a, b in zip(words_t, words_b, strict=False)]):.3f} (words stay aligned only while no space is lost)"
)
for k in (0, 1, 2):
    print("original :", repr(test_text[k * 64 : k * 64 + 64]))
    print("rebuilt  :", repr(back[k * 64 : k * 64 + 64]))

## Slide: The Command Line (C1)

The model of the notebook is saved; the three commands of `python -m faa.textae` then run in a separate process: `compress` writes the binary file, `decompress` rebuilds the text. The result must equal the in-process result above.

In [ ]:
ae.save(work / "textae.keras")
run = [sys.executable, "-m", "faa.textae"]
r1 = subprocess.run(
    [*run, "compress", str(work / "test.txt"), str(work / "test.bin"), "-m", str(work / "textae.keras")],
    capture_output=True,
    text=True,
    check=True,
)
r2 = subprocess.run(
    [*run, "decompress", str(work / "test.bin"), str(work / "back.txt"), "-m", str(work / "textae.keras")],
    capture_output=True,
    text=True,
    check=True,
)
slide("The Command Line")
print(r1.stdout.strip())
print(r2.stdout.strip())
cli_back = (work / "back.txt").read_text(encoding="utf-8")
print(f"binary file: {(work / 'test.bin').stat().st_size} bytes; same text as in-process: {cli_back == back}")

## Slide: Lossy Against Lossless (C2)

The same 20 000 characters with general-purpose compressors (lossless: the text comes back exactly) and with the autoencoder (lossy).

In [ ]:
raw = test_text.encode()
rows = [
    {"method": "gzip -9", "bytes": len(gzip.compress(raw, 9)), "exact": True},
    {"method": "bzip2 -9", "bytes": len(bz2.compress(raw, 9)), "exact": True},
    {"method": "lzma", "bytes": len(lzma.compress(raw, preset=9)), "exact": True},
    {"method": "autoencoder", "bytes": len(blob), "exact": False},
]
for r in rows:
    r["ratio"] = len(raw) / r["bytes"]
cmp = pl.DataFrame(rows)
slide("Lossy Against Lossless")
show(cmp)
print(
    f"character accuracy of the autoencoder: {textae.char_accuracy(test_text, back):.3f}; the others return the text exactly"
)

## Slide: How Big Must the Code Be? (C2)

Three code sizes, 4 bits per unit, trained for 5 epochs on 20 000 windows (a short run, so the numbers are lower than above): the bytes per window and the share of characters rebuilt on the unseen file.

In [ ]:
rows = []
X2 = textae.windows(train_texts, 20000, 1)
for code in (8, 16, 32):
    a2, e2, d2 = textae.build(code, 64, 1, "conv", 4)
    a2.fit(X2, X2, epochs=5, batch_size=128, shuffle=True, verbose=0)
    b2 = textae.compress(test_text, e2, 4)
    rows.append(
        {
            "code_units": code,
            "bytes_per_window": code // 2,
            "ratio": len(raw) / len(b2),
            "char_accuracy": textae.char_accuracy(test_text, textae.decompress(b2, d2)),
        }
    )
sizes = pl.DataFrame(rows)
slide("How Big Must the Code Be?")
show(sizes)

fig, ax = plotlib.fig("col", 1.6)
ax.plot(sizes["bytes_per_window"], sizes["char_accuracy"], color=BLUE, marker="o", ms=3)
for r in sizes.iter_rows(named=True):
    ax.annotate(
        f"{r['ratio']:.1f}:1",
        (r["bytes_per_window"], r["char_accuracy"]),
        fontsize=6,
        xytext=(3, -8),
        textcoords="offset points",
    )
ax.set_xlabel("bytes per window of 32 characters")
ax.set_ylabel("characters rebuilt")
plotlib.save(fig, "c2_code_size")

## Failure case: text the training files never showed

The same compressor on a Python source file, on a Markdown page of this repository and on random characters. The autoencoder learned the statistics of English news.

In [ ]:
root = Path(data.data_dir()).parent
others = {
    "unseen news (entertainment)": test_text,
    "Markdown page (practice/10_clustering.md)": (root / "practice/10_clustering.md").read_text(encoding="utf-8")[
        :20000
    ],
    "Python source (faa/clustering.py)": (root / "src/faa/clustering.py").read_text(encoding="utf-8")[:20000],
    "random characters": "".join(np.random.default_rng(0).choice(list(textae.CHARS[1:]), 20000)),
}
rows = []
for name, t in others.items():
    bb = textae.compress(t, enc, 4)
    rows.append(
        {
            "text": name,
            "ratio": len(t.encode()) / len(bb),
            "char_accuracy": textae.char_accuracy(t, textae.decompress(bb, dec)),
        }
    )
slide("Text the Training Files Never Showed")
show(pl.DataFrame(rows))